# Workflow for a single node energy system

In this application of the ETHOS.FINE framework, a single region energy system is modeled and optimized.

All classes which are available to the user are utilized and examples of the selection of different parameters within these classes are given.

The workflow is structures as follows:
1. Required packages are imported and the input data path is set
2. An energy system model instance is created
3. Commodity sources are added to the energy system model
4. Commodity conversion components are added to the energy system model
5. Commodity storages are added to the energy system model
6. Commodity sinks are added to the energy system model
7. Material sinks and sources are added to the energy system model
8. The energy system model is optimized
9. Selected optimization results are presented


# 1. Import required packages and set input data path

The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [1]:
import fine as fn
from getData import getData
from pathlib import Path
import pandas as pd


cwd = Path.cwd()
data = getData()

%matplotlib inline
%load_ext autoreload
%autoreload 2

# 2. Create an energy system model instance 

The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [2]:
locations = {"GermanyRegion"}
commodityUnitsDict = {"electricity": r"GW$_{el}$", "hydrogen": r"GW$_{H_{2},LHV}$"}
commodities = {"electricity", "hydrogen"}
materials = {"steel", "copper"}
materialUnitsDict = {"steel": r"tons$", "copper": r"tons$"}
numberOfTimeSteps = 8760
hoursPerTimeStep = 1

In [3]:
initial_material_cost = {
    "steel": pd.Series(
        {
            "GermanyRegion": 0.1,
        }
    ),
    "copper": pd.Series(
        {
            "GermanyRegion": 0.1,
        }
    ),
}

In [4]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    materials=materials,
    numberOfTimeSteps=8760,
    commodityUnitsDict=commodityUnitsDict,
    materialUnitsDict=materialUnitsDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
    initialMaterialCost=initial_material_cost,
)

# 3. Add commodity sources to the energy system model

## 3.1. Electricity sources

### Wind onshore

In [5]:
esM.add(
    fn.Source(
        esM=esM,
        name="Wind (onshore)",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=1.1,
        opexPerCapacity=1.1 * 0.02,
        interestRate=0.08,
        economicLifetime=20,
        materialIntensity={
            0: {
                "steel": pd.Series({"GermanyRegion": 5.2}),
                "copper": pd.Series({"GermanyRegion": 4.2}),
            },
        },
    )
)

Full load hours:

In [6]:
data["Wind (onshore), operationRateMax"].sum()

2300.4069071646272

# 4. Add conversion components to the energy system model

### New combined cycly gas turbines for hydrogen

In [7]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="New CCGT plants (hydrogen)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": 1, "hydrogen": -1 / 0.6},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
        materialIntensity={
            0: {
                "steel": pd.Series({"GermanyRegion": 3.1}),
                "copper": pd.Series({"GermanyRegion": 2.1}),
            },
        },
    )
)

### Electrolyzers

In [8]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electroylzers",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=10,
    )
)

# 5. Add commodity storages to the energy system model

## 5.1. Electricity storage

### Lithium ion batteries

The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [9]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
        materialIntensity={
            0: {
                "steel": pd.Series({"GermanyRegion": 5.0}),
                "copper": pd.Series({"GermanyRegion": 2.0}),
            },
        },
    )
)

## 5.2. Hydrogen storage

### Hydrogen filled salt caverns
The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

In [10]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (hydrogen)",
        commodity="hydrogen",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=133,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (hydrogen), capacityMax"],
        investPerCapacity=0.00011,
        opexPerCapacity=0.00057,
        interestRate=0.08,
        economicLifetime=30,
    )
)

# 6. Add commodity sinks to the energy system model

## 6.1. Electricity sinks

### Electricity demand

In [11]:
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=data["Electricity demand, operationRateFix"],
    )
)

## 6.2. Hydrogen sinks

### Fuel cell electric vehicle (FCEV) demand

In [12]:
FCEV_penetration = 0.5
esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=data["Hydrogen demand, operationRateFix"] * FCEV_penetration,
    )
)

# 7. Add material sinks and sources

## 7.1 Material sinks

In [13]:
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        hasCapacityVariable=False,
        commodity="steel",
        material=True,
    )
)

sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Copper demand",
        hasCapacityVariable=False,
        commodity="copper",
        material=True,
    )
)

## 7.2 Material sources

In [14]:
esM.add(
    fn.Source(
        esM=esM,
        name="Steel supply",
        hasCapacityVariable=False,
        commodity="steel",
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Copper supply",
        hasCapacityVariable=False,
        commodity="copper",
    )
)

# 8. Optimize energy system model

All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

In [15]:
esM.aggregateTemporally(numberOfTypicalPeriods=30)


Clustering time series data with 30 typical periods and 24 time steps per period 
further clustered to 12 segments per period...
		(3.5637 sec)



In [16]:
esM.optimize(timeSeriesAggregation=True, solver="gurobi")

Time series aggregation specifications:
Number of typical periods:30, number of time steps per period:24, number of segments per period:12

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.1614 sec)

Declaring sets, variables and constraints for ConversionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0519 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.4506 sec)

		(0.0002 sec)

Declaring shared potential constraint...
		(0.0010 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.2049 sec)

Declaring material demand constraints...
RHS_demand 5.2*commis_srcSnk[GermanyRegion,'Wind (onshore)',0] + 3.1*commis_conv[GermanyRegion,'New CCGT plants (hydrogen)',0] + 5.0*commis_stor[GermanyRegion,Li-ion batteries,0]
RHS_de

# 9. Selected results output

### Sources and Sink

Show optimization summary

In [22]:
total_cost = 0.0

for loc, mat in esM.pyM.initialMaterialSet:
    quantity = esM.pyM.initialMaterialSupply[loc, mat].value
    unit_cost = esM.initialMaterialCost[mat][loc]
    material_cost = quantity * unit_cost

    print(f"{mat}:")
    print(f"  Initial supply: {quantity:.6f} [t]")
    print(f"  Unit cost: {unit_cost:.8f} [USD/t]")
    print(f"  Cost contribution: {material_cost:.6f} [USD]")

    total_cost += material_cost

print(f"\nTotal initial material cost: {total_cost:.6f} [USD]")

steel:
  Initial supply: 404.810060 [t]
  Unit cost: 0.10000000 [USD/t]
  Cost contribution: 40.481006 [USD]
copper:
  Initial supply: 268.719110 [t]
  Unit cost: 0.10000000 [USD/t]
  Cost contribution: 26.871911 [USD]

Total initial material cost: 67.352917 [USD]


In [18]:
import pyomo.environ as pyomo

initial_cost = sum(
    esM.pyM.initialMaterialSupply[loc, mat] * esM.initialMaterialCost[mat][loc]
    for loc, mat in esM.pyM.initialMaterialSet
)

print("Initial material cost:", pyomo.value(initial_cost))
print("Total Objective:", pyomo.value(esM.pyM.Obj))

Initial material cost: 67.35291707081589
Total Objective: 75.61657566125133


In [19]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

GermanyRegion
Component          Property        Unit                                
Electricity demand operation       [GW$_{el}$*h/a]         30957.888055
                                   [GW$_{el}$*h]           30957.888055
Hydrogen demand    operation       [GW$_{H_{2},LHV}$*h/a]   4765.074877
                                   [GW$_{H_{2},LHV}$*h]     4765.074877
Wind (onshore)     NPVcontribution [1e9 Euro]                   6.66911
                   TAC             [1e9 Euro/a]                 6.66911
                   capacity        [GW$_{el}$]                49.755577
                   capexCap        [1e9 Euro/a]                5.574487
                   commissioning   [GW$_{el}$]                49.755577
                   invest          [1e9 Euro]                 54.731135
                   operation       [GW$_{el}$*h/a]         42634.829474
                                   [GW$_{el}$*h]           42634.829474
                   opexCap         [1e9 Euro/a]                1.094623

Plot operation time series (either one or two dimensional)

### Conversion

Show optimization summary

In [20]:
esM.getOptimizationSummary("ConversionModel", outputLevel=1)

GermanyRegion
Component                  Property                        Unit                         
Electroylzers              NPVcontribution                 [1e9 Euro]           0.278493
                           TAC                             [1e9 Euro/a]         0.278493
                           capacity                        [GW$_{el}$]          3.200523
                           capexCap                        [1e9 Euro/a]         0.238486
                           commissioning                   [GW$_{el}$]          3.200523
                           decommissioning                 [GW$_{el}$]               0.0
                           invest                          [1e9 Euro]           1.600262
                           investLifetimeExtension         [1e9 Euro]                  0
                           operation                       [GW$_{el}$*h/a]  14331.232349
                                                           [GW$_{el}$*h]    14331.232349
                           opexCap                         [1e9 Euro/a]         0.040007
                           opexOp                          [1e9 Euro/a]              0.0
                           revenueLifetimeShorteningResale [1e9 Euro]                  0
New CCGT plants (hydrogen) NPVcontribution                 [1e9 Euro]           0.124907
                           TAC                             [1e9 Euro/a]         0.124907
                           capacity                        [GW$_{el}$]          1.527049
                           capexCap                        [1e9 Euro/a]         0.092839
                           commissioning                   [GW$_{el}$]          1.527049
                           decommissioning                 [GW$_{el}$]               0.0
                           invest                          [1e9 Euro]           1.068934
                           investLifetimeExtension         [1e9 Euro]                  0
                           operation                       [GW$_{el}$*h/a]    3160.07266
                                                           [GW$_{el}$*h]      3160.07266
                           opexCap                         [1e9 Euro/a]         0.032068
                           opexOp                          [1e9 Euro/a]              0.0
                           revenueLifetimeShorteningResale [1e9 Euro]                  0

### Storage

Show optimization summary

In [21]:
esM.getOptimizationSummary("StorageModel", outputLevel=2)

GermanyRegion
Component               Property           Unit                                
Li-ion batteries        NPVcontribution    [1e9 Euro]                  0.475007
                        TAC                [1e9 Euro/a]                0.475007
                        capacity           [GW$_{el}$*h]              28.269441
                        capexCap           [1e9 Euro/a]                0.418468
                        commissioning      [GW$_{el}$*h]              28.269441
                        invest             [1e9 Euro]                  4.268686
                        operationCharge    [GW$_{el}$*h/a]          5137.673084
                                           [GW$_{el}$*h]            5137.673084
                        operationDischarge [GW$_{el}$*h/a]          4631.891352
                                           [GW$_{el}$*h]            4631.891352
                        opexCap            [1e9 Euro/a]                0.056539
Salt caverns (hydrogen) NPVcontribution    [1e9 Euro]                  0.716143
                        TAC                [1e9 Euro/a]                0.716143
                        capacity           [GW$_{H_{2},LHV}$*h]     1235.216115
                        capexCap           [1e9 Euro/a]                0.012069
                        commissioning      [GW$_{H_{2},LHV}$*h]     1235.216115
                        invest             [1e9 Euro]                  0.135874
                        operationCharge    [GW$_{H_{2},LHV}$*h/a]   7465.282234
                                           [GW$_{H_{2},LHV}$*h]     7465.282234
                        operationDischarge [GW$_{H_{2},LHV}$*h/a]   7465.282234
                                           [GW$_{H_{2},LHV}$*h]     7465.282234
                        opexCap            [1e9 Euro/a]                0.704073